# Evaluación Formativa 1: caracterización inicial del dataset Framingham

**Asignatura:** Estadística Computacional  
**Autor:** Alexander Sepúlveda Ormazábal  
**Modalidad:** Trabajo individual  
**Dataset:** Framingham Heart Study  

## Objetivo

Explorar un conjunto de datos de salud cardiovascular, identificar y clasificar sus variables, describir su comportamiento y estudiar relaciones preliminares relevantes para la evaluación del riesgo de enfermedad coronaria a diez años.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

print("Python utilizado:")
print(sys.executable)
print("\nVersión de pandas:", pd.__version__)

Python utilizado:
C:\MCDI501-Framingham\.venv\Scripts\python.exe

Versión de pandas: 3.0.6


In [3]:
# Ruta relativa desde la carpeta principal del repositorio
RUTA_DATOS = Path("fase-1/datos/raw/framingham.csv")

print("Carpeta de trabajo actual:")
print(Path.cwd())

if not RUTA_DATOS.exists():
    raise FileNotFoundError(
        f"No se encontró el dataset en: {RUTA_DATOS.resolve()}"
    )

df = pd.read_csv(RUTA_DATOS)

print("\nDataset cargado correctamente.")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")



Carpeta de trabajo actual:
C:\MCDI501-Framingham

Dataset cargado correctamente.
Filas: 4238
Columnas: 16


In [4]:
# Mostrar las primeras cinco observaciones
display(df.head())

print("\nInformación general del dataset:\n")
df.info()

# Resumen de tipos, valores ausentes y valores distintos
resumen_calidad = pd.DataFrame({
    "tipo_python": df.dtypes.astype(str),
    "valores_no_nulos": df.notna().sum(),
    "valores_nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(dropna=True)
})

print("\nResumen de calidad de los datos:")
display(resumen_calidad)

print("\nFilas duplicadas:", df.duplicated().sum())

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0



Información general del dataset:

<class 'pandas.DataFrame'>
RangeIndex: 4238 entries, 0 to 4237
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             4238 non-null   int64  
 1   age              4238 non-null   int64  
 2   education        4133 non-null   float64
 3   currentSmoker    4238 non-null   int64  
 4   cigsPerDay       4209 non-null   float64
 5   BPMeds           4185 non-null   float64
 6   prevalentStroke  4238 non-null   int64  
 7   prevalentHyp     4238 non-null   int64  
 8   diabetes         4238 non-null   int64  
 9   totChol          4188 non-null   float64
 10  sysBP            4238 non-null   float64
 11  diaBP            4238 non-null   float64
 12  BMI              4219 non-null   float64
 13  heartRate        4237 non-null   float64
 14  glucose          3850 non-null   float64
 15  TenYearCHD       4238 non-null   int64  
dtypes: float64(9), int64(7)
memory usage

,tipo_python,valores_no_nulos,valores_nulos,porcentaje_nulos,valores_unicos
male,int64,4238,0,0.00,2
age,int64,4238,0,0.00,39
education,float64,4133,105,2.48,4
currentSmoker,int64,4238,0,0.00,2
cigsPerDay,float64,4209,29,0.68,33
BPMeds,float64,4185,53,1.25,2
prevalentStroke,int64,4238,0,0.00,2
prevalentHyp,int64,4238,0,0.00,2
diabetes,int64,4238,0,0.00,2
totChol,float64,4188,50,1.18,248



Filas duplicadas: 0


## 1. Contexto y problema de análisis

Las enfermedades cardiovasculares constituyen un problema relevante de salud pública. Su aparición depende de una combinación de características demográficas, hábitos y condiciones clínicas. Analizar estos factores permite describir perfiles de riesgo y apoyar decisiones preventivas bajo incertidumbre.

Este trabajo utiliza una versión del conjunto de datos del **Framingham Heart Study**, obtenida desde [Kaggle](https://www.kaggle.com/datasets/dileep070/heart-disease-prediction-using-logistic-regression). El archivo contiene 4.238 observaciones y 16 variables relacionadas con características demográficas, tabaquismo, presión arterial, colesterol, diabetes y otros indicadores de salud.

La variable de resultado es `TenYearCHD`, que indica si la persona desarrolló enfermedad coronaria durante un periodo de diez años:

- `0`: no desarrolló enfermedad coronaria.
- `1`: desarrolló enfermedad coronaria.

### Pregunta de análisis

### Pregunta de análisis

¿Qué diferencias demográficas, conductuales y clínicas se observan entre las personas que desarrollaron y las que no desarrollaron enfermedad coronaria durante los diez años de seguimiento?

### Fuentes

- [Dataset utilizado en Kaggle](https://www.kaggle.com/datasets/dileep070/heart-disease-prediction-using-logistic-regression)
- [Documentación del Framingham Heart Study — NHLBI](https://biolincc.nhlbi.nih.gov/teaching/)

In [5]:
diccionario_variables = pd.DataFrame([
    ["male", "Sexo registrado", "Cualitativa nominal dicotómica",
     "0 = mujer; 1 = hombre", "Explicativa"],

    ["age", "Edad de la persona", "Cuantitativa discreta",
     "Años cumplidos", "Explicativa"],

    ["education", "Nivel educacional alcanzado", "Cualitativa ordinal",
     "1 = 0–11 años; 2 = enseñanza media/GED; 3 = educación técnica o universitaria incompleta; 4 = título universitario o superior",
     "Explicativa"],

    ["currentSmoker", "Condición actual de fumador", "Cualitativa nominal dicotómica",
     "0 = no; 1 = sí", "Explicativa"],

    ["cigsPerDay", "Cantidad de cigarrillos consumidos diariamente",
     "Cuantitativa discreta", "Cigarrillos por día", "Explicativa"],

    ["BPMeds", "Uso de medicamentos antihipertensivos",
     "Cualitativa nominal dicotómica", "0 = no; 1 = sí", "Explicativa"],

    ["prevalentStroke", "Antecedente de accidente cerebrovascular",
     "Cualitativa nominal dicotómica", "0 = no; 1 = sí", "Explicativa"],

    ["prevalentHyp", "Diagnóstico previo de hipertensión",
     "Cualitativa nominal dicotómica", "0 = no; 1 = sí", "Explicativa"],

    ["diabetes", "Diagnóstico de diabetes",
     "Cualitativa nominal dicotómica", "0 = no; 1 = sí", "Explicativa"],

    ["totChol", "Colesterol total", "Cuantitativa continua",
     "mg/dL", "Explicativa"],

    ["sysBP", "Presión arterial sistólica", "Cuantitativa continua",
     "mmHg", "Explicativa"],

    ["diaBP", "Presión arterial diastólica", "Cuantitativa continua",
     "mmHg", "Explicativa"],

    ["BMI", "Índice de masa corporal", "Cuantitativa continua",
     "kg/m²", "Explicativa"],

    ["heartRate", "Frecuencia cardíaca", "Cuantitativa continua",
     "Latidos por minuto", "Explicativa"],

    ["glucose", "Concentración de glucosa", "Cuantitativa continua",
     "mg/dL", "Explicativa"],

    ["TenYearCHD", "Aparición de enfermedad coronaria en diez años",
     "Cualitativa nominal dicotómica", "0 = no; 1 = sí", "Respuesta"]
], columns=[
    "variable",
    "descripcion",
    "tipo_estadistico",
    "unidad_o_codificacion",
    "rol"
])

# Comprobar que el diccionario cubra exactamente las columnas del dataset
columnas_dataset = set(df.columns)
columnas_documentadas = set(diccionario_variables["variable"])

assert columnas_dataset == columnas_documentadas, (
    "El diccionario y las columnas del dataset no coinciden. "
    f"Sin documentar: {columnas_dataset - columnas_documentadas}. "
    f"No encontradas: {columnas_documentadas - columnas_dataset}."
)

print("Las 16 variables están documentadas correctamente.")
display(diccionario_variables)

Las 16 variables están documentadas correctamente.


,variable,descripcion,tipo_estadistico,unidad_o_codificacion,rol
0,male,Sexo registrado,Cualitativa nominal dicotómica,0 = mujer; 1 = hombre,Explicativa
1,age,Edad de la persona,Cuantitativa discreta,Años cumplidos,Explicativa
2,education,Nivel educacional alcanzado,Cualitativa ordinal,1 = 0–11 años; 2 = enseñanza media/GED; 3 = ed...,Explicativa
3,currentSmoker,Condición actual de fumador,Cualitativa nominal dicotómica,0 = no; 1 = sí,Explicativa
4,cigsPerDay,Cantidad de cigarrillos consumidos diariamente,Cuantitativa discreta,Cigarrillos por día,Explicativa
5,BPMeds,Uso de medicamentos antihipertensivos,Cualitativa nominal dicotómica,0 = no; 1 = sí,Explicativa
6,prevalentStroke,Antecedente de accidente cerebrovascular,Cualitativa nominal dicotómica,0 = no; 1 = sí,Explicativa
7,prevalentHyp,Diagnóstico previo de hipertensión,Cualitativa nominal dicotómica,0 = no; 1 = sí,Explicativa
8,diabetes,Diagnóstico de diabetes,Cualitativa nominal dicotómica,0 = no; 1 = sí,Explicativa
9,totChol,Colesterol total,Cuantitativa continua,mg/dL,Explicativa


### Interpretación de los tipos de variables

Aunque todas las columnas están almacenadas mediante números, no todas representan cantidades. Las variables `male`, `currentSmoker`, `BPMeds`, `prevalentStroke`, `prevalentHyp`, `diabetes` y `TenYearCHD` utilizan 0 y 1 como códigos de categorías.

`education` es una variable cualitativa ordinal: sus niveles tienen un orden, pero las distancias entre niveles consecutivos no representan diferencias educativas equivalentes.

`age` y `cigsPerDay` son cuantitativas discretas porque están registradas mediante valores contables. Las variables clínicas `totChol`, `sysBP`, `diaBP`, `BMI`, `heartRate` y `glucose` son cuantitativas continuas porque representan mediciones.

### Calidad inicial de los datos

El dataset contiene 4.238 observaciones y no presenta filas duplicadas. Se identificaron valores ausentes en siete variables. `glucose` presenta la mayor ausencia, con aproximadamente 9,16 %, mientras que las demás variables tienen menos de 2,5 % de valores ausentes.

En esta etapa no se eliminarán ni reemplazarán observaciones. Para cada análisis se utilizarán los registros disponibles en las variables involucradas y se informará el tamaño muestral efectivo. Esta decisión evita introducir valores artificiales y conserva la mayor cantidad posible de información.